In [1]:
import pandas as pd
import nfl_data_py as nfl

In [7]:
from data.data_aggregation import import_pbp_data

In [34]:
df = pd.read_pickle('pbp_data.pkl')

In [15]:
ot_games = df[df['game_half'] == 'Overtime']['game_id'].unique()

array(['2025_02_NYG_DAL', '2025_04_GB_DAL', '2025_05_SF_LA',
       '2025_09_JAX_LV', '2025_10_ATL_IND', '2025_11_CAR_ATL',
       '2025_11_WAS_MIA', '2025_12_IND_KC', '2025_12_JAX_ARI',
       '2025_12_NYG_DET', '2025_13_DEN_WAS', '2025_14_PHI_LAC',
       '2025_16_GB_CHI', '2025_16_LA_SEA', '2025_20_BUF_DEN',
       '2025_20_LA_CHI', '2024_01_LA_DET', '2024_02_SEA_NE',
       '2024_05_BAL_CIN', '2024_05_TB_ATL', '2024_09_LA_SEA',
       '2024_09_NE_TEN', '2024_09_TB_KC', '2024_10_NYG_CAR',
       '2024_12_MIN_CHI', '2024_13_TB_CAR', '2024_14_NYJ_MIA',
       '2024_16_ARI_CAR', '2024_17_ATL_WAS', '2024_17_DEN_CIN',
       '2024_18_CAR_ATL', '2024_18_JAX_IND', '2023_01_BUF_NYJ',
       '2023_02_LAC_TEN', '2023_02_SEA_DET', '2023_03_IND_BAL',
       '2023_04_LA_IND', '2023_04_WAS_PHI', '2023_08_NYJ_NYG',
       '2023_12_BUF_PHI', '2023_13_CIN_JAX', '2023_13_IND_TEN',
       '2023_14_LA_BAL', '2023_15_HOU_TEN', '2023_15_MIN_CIN',
       '2023_22_SF_KC', '2022_01_IND_HOU', '2022_01_PIT_CI

In [18]:
df_ot = df[df['game_half'] == 'Overtime'].sort_values(by = ['game_id', 'play_id'])

In [27]:
import pandas as pd

def get_ruleset(season, week):
    is_playoff = week > 18
    if season >= 2025:
        return 'post2025'
    elif season >= 2022 and is_playoff:
        return 'post2025'  # same rules as post2025
    elif season >= 2012:
        return '2012_2024'
    else:
        return 'pre2012'

def aggregate_overtime_games(pbp: pd.DataFrame) -> pd.DataFrame:
    ot_plays = pbp[pbp['game_half'] == 'Overtime'].copy()

    results = []

    for game_id, game in ot_plays.groupby('game_id'):
        game = game.sort_values('play_id')

        first_play = game.iloc[0]
        season = first_play['season']
        week = first_play['week']
        receiving_team = first_play['posteam']

        home = first_play['home_team']
        away = first_play['away_team']
        kicking_team = away if receiving_team == home else home

        last_play = game.iloc[-1]
        home_score = last_play['total_home_score']
        away_score = last_play['total_away_score']

        if home_score > away_score:
            winner = home
        elif away_score > home_score:
            winner = away
        else:
            winner = 'TIE'

        results.append({
            'game_id': game_id,
            'season': season,
            'week': week,
            'ruleset': get_ruleset(season, week),
            'kicking_team': kicking_team,
            'receiving_team': receiving_team,
            'winner': winner,
            'kicking_team_won': winner == kicking_team,
            'receiving_team_won': winner == receiving_team,
            'tie': winner == 'TIE',
        })

    return pd.DataFrame(results).sort_values(['season', 'week']).reset_index(drop=True)

In [35]:
ot_df = aggregate_overtime_games(df)

In [36]:
ot_df.groupby('ruleset').agg(
    games=('game_id', 'count'),
    kicking_won=('kicking_team_won', 'sum'),
    receiving_won=('receiving_team_won', 'sum'),
    ties=('tie', 'sum'),
    kicking_pct=('kicking_team_won', 'mean'),
    receiving_pct=('receiving_team_won', 'mean'),
    tie_pct=('tie', 'mean'),
).round(3).T


ruleset,2012_2024,post2025,pre2012
games,211.000,17.000,93.000
kicking_won,83.000,7.000,43.000
receiving_won,113.000,9.000,49.000
ties,15.000,1.000,1.000
kicking_pct,0.393,0.412,0.462
receiving_pct,0.536,0.529,0.527
tie_pct,0.071,0.059,0.011
